# Function Calling Fundamentals

**Phase 03** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-03/01-function-calling-fundamentals.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic pydantic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 03-01: Function Calling Fundamentals
Raw tool dispatch loop using only the anthropic SDK. No frameworks.

Run: python main.py
Run with a specific question: python main.py --question "What are my recent transactions for acc_42?"
"""

import argparse
import json

import anthropic

### Tool schemas (raw dicts, the format the Anthropic API expects)

In [ ]:
TOOLS = [
    {
        "name": "get_account_balance",
        "description": (
            "Returns the current balance for a given account. "
            "Use this when the user asks about their balance, funds, or how much money is in an account."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "account_id": {
                    "type": "string",
                    "description": "The account identifier, e.g. 'acc_42' or 'acc_7891'.",
                }
            },
            "required": ["account_id"],
        },
    },
    {
        "name": "list_recent_transactions",
        "description": (
            "Returns the N most recent transactions for an account. "
            "Use this when the user asks about recent activity, charges, deposits, or spending history."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "account_id": {
                    "type": "string",
                    "description": "The account identifier.",
                },
                "limit": {
                    "type": "integer",
                    "description": "Number of transactions to return. Defaults to 5. Maximum 20.",
                    "default": 5,
                },
            },
            "required": ["account_id"],
        },
    },
]

### Stub functions (replace with real DB queries in production)

In [ ]:
def get_account_balance(account_id: str) -> dict:
    """
    Stub: returns a realistic fake balance.
    Production: return db.query("SELECT balance FROM accounts WHERE id = ?", account_id)
    """
    stub_data = {
        "acc_42":   {"balance": 14.22,    "currency": "USD", "account_id": "acc_42"},
        "acc_99":   {"balance": 8_204.50, "currency": "USD", "account_id": "acc_99"},
        "acc_7891": {"balance": 0.00,     "currency": "USD", "account_id": "acc_7891"},
    }
    if account_id not in stub_data:
        return {"error": f"Account {account_id!r} not found.", "account_id": account_id}
    return stub_data[account_id]


def list_recent_transactions(account_id: str, limit: int = 5) -> dict:
    """
    Stub: returns realistic fake transaction history.
    Production: return db.query("SELECT ... FROM transactions WHERE account_id = ? LIMIT ?", ...)
    """
    stub_transactions: dict[str, list[dict]] = {
        "acc_42": [
            {"date": "2026-05-24", "description": "Coffee Shop",       "amount": -4.50,    "type": "debit"},
            {"date": "2026-05-23", "description": "Payroll Deposit",   "amount": 2_000.00, "type": "credit"},
            {"date": "2026-05-22", "description": "Grocery Store",     "amount": -87.33,   "type": "debit"},
            {"date": "2026-05-21", "description": "Streaming Service", "amount": -15.99,   "type": "debit"},
            {"date": "2026-05-20", "description": "ATM Withdrawal",    "amount": -60.00,   "type": "debit"},
        ],
        "acc_99": [
            {"date": "2026-05-24", "description": "Wire Transfer In",  "amount": 5_000.00, "type": "credit"},
            {"date": "2026-05-22", "description": "Online Purchase",   "amount": -129.99,  "type": "debit"},
        ],
    }
    txns = stub_transactions.get(account_id, [])
    return {
        "account_id": account_id,
        "transactions": txns[:limit],
        "count": min(limit, len(txns)),
    }

### Dispatch

In [ ]:
FUNCTION_MAP = {
    "get_account_balance": get_account_balance,
    "list_recent_transactions": list_recent_transactions,
}


def dispatch_tool_call(tool_name: str, tool_input: dict) -> str:
    """Look up and call the right function. Returns a JSON string."""
    if tool_name not in FUNCTION_MAP:
        return json.dumps({"error": f"Unknown tool: {tool_name!r}"})
    result = FUNCTION_MAP[tool_name](**tool_input)
    return json.dumps(result)

### Tool dispatch loop

In [ ]:
def run_with_tools(user_message: str, verbose: bool = True) -> str:
    """
    Full two-round-trip tool-use dispatch loop.

    Round 1: Send user message + tool schemas. LLM returns tool_use block.
    Round 2: Execute tools, send results. LLM returns final text answer.
    """
    client = anthropic.Anthropic()
    messages: list[dict] = [{"role": "user", "content": user_message}]

    if verbose:
        print(f"\n[user] {user_message}")

    # --- Round 1: get tool call request ---
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=1024,
        tools=TOOLS,
        messages=messages,
    )

    if verbose:
        print(f"[api]  stop_reason={response.stop_reason}")

    # Model answered directly without needing a tool.
    if response.stop_reason == "end_turn":
        final_text = response.content[0].text
        if verbose:
            print(f"[assistant] {final_text}")
        return final_text

    # Collect tool_use blocks.
    tool_uses = [block for block in response.content if block.type == "tool_use"]

    if not tool_uses:
        # Unexpected: fall through to text.
        text = next((b.text for b in response.content if hasattr(b, "text")), "")
        return text

    # Append the full assistant response (tool_use blocks included) to message history.
    messages.append({"role": "assistant", "content": response.content})

    # Execute each tool and collect results.
    tool_results = []
    for tool_use in tool_uses:
        if verbose:
            print(f"[tool] {tool_use.name}({json.dumps(tool_use.input)})")
        result_str = dispatch_tool_call(tool_use.name, tool_use.input)
        if verbose:
            print(f"[result] {result_str[:120]}")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": result_str,
        })

    # Append all results as a single user turn.
    messages.append({"role": "user", "content": tool_results})

    # --- Round 2: get final natural-language answer ---
    final_response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=1024,
        tools=TOOLS,
        messages=messages,
    )

    final_text = next(
        (b.text for b in final_response.content if hasattr(b, "text")), ""
    )
    if verbose:
        print(f"[assistant] {final_text}")
    return final_text

### Pydantic schema generation (USE IT section)

In [ ]:
def pydantic_schema_demo() -> None:
    """
    Demonstrates generating Claude-compatible tool schemas from Pydantic models.
    No API call is made; this just prints the generated schema.
    """
    try:
        from pydantic import BaseModel, Field
        from functools import wraps
    except ImportError:
        print("Install pydantic: pip install pydantic")
        return

    class GetAccountBalanceInput(BaseModel):
        account_id: str = Field(
            description="The account identifier, e.g. 'acc_42' or 'acc_7891'."
        )

    class ListRecentTransactionsInput(BaseModel):
        account_id: str = Field(description="The account identifier.")
        limit: int = Field(
            default=5, ge=1, le=20,
            description="Number of transactions to return. Defaults to 5. Maximum 20."
        )

    def make_tool_schema(name: str, description: str, input_model: type) -> dict:
        schema = input_model.model_json_schema()
        schema.pop("title", None)
        return {"name": name, "description": description, "input_schema": schema}

    tools_from_pydantic = [
        make_tool_schema(
            "get_account_balance",
            "Returns the current balance for a given account.",
            GetAccountBalanceInput,
        ),
        make_tool_schema(
            "list_recent_transactions",
            "Returns the N most recent transactions for an account.",
            ListRecentTransactionsInput,
        ),
    ]

    print("\n=== Pydantic-generated schemas ===")
    print(json.dumps(tools_from_pydantic, indent=2))

### Main

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="Lesson 03-01: Function Calling Fundamentals")
    parser.add_argument(
        "--question",
        default="What's my account balance for account acc_42?",
        help="The user question to send.",
    )
    parser.add_argument(
        "--pydantic",
        action="store_true",
        help="Show Pydantic schema generation instead of making an API call.",
    )
    args = parser.parse_args()

    if args.pydantic:
        pydantic_schema_demo()
        return

    print("=== 03-01: Function Calling Fundamentals ===")
    print("Tools registered:", list(FUNCTION_MAP.keys()))

    answer = run_with_tools(args.question)
    print(f"\nFinal answer:\n{answer}")

    # Second demo: transactions
    print("\n" + "=" * 50)
    run_with_tools(
        "Show me the last 3 transactions for account acc_42.",
    )

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. Why did the bot give a wrong answer despite having a tool schema defined?**

- A. The tool schema description was too vague, so the LLM chose not to call it
- B. The LLM executed the function itself and returned a hallucinated result because no real function was wired to the schema
- C. The schema was correct, but the LLM generated a JSON tool_use block pointing to a function that was never dispatched, so the LLM answered from its prior knowledge instead
- D. The LLM called the function, but the wrong order ID was passed in the input

**2. What is missing from the messages list?**

- A. A system prompt at the start of the messages list
- B. The assistant's response containing the tool_use block, which must be inserted between the user message and the tool_result
- C. The tool_result must be in its own separate API call, not combined with previous messages
- D. The tool_result content must be plain text, not a JSON string

**3. What is the most direct cause of the LLM using wrong parameter names?**

- A. The model being used is not capable of respecting tool schemas
- B. The parameter names in the input_schema properties do not match what you describe in the tool description, causing the LLM to infer its own names
- C. The parameter names in the tool schema itself use 'query' and 'max_results', but the description or examples somewhere in the schema or system prompt use 'q' and 'n', confusing the LLM
- D. The LLM always generates abbreviated parameter names for efficiency

**4. Beyond just 'option B is better,' what are the two most important production reasons to prefer the tool approach?**

- A. Tools are faster because the LLM doesn't have to read the entire JSON blob; and tools let you add rate limiting and caching at the dispatch layer
- B. Tools are cheaper per token; and tools prevent the LLM from reading sensitive data it shouldn't see
- C. The JSON blob in the system prompt would be stale as soon as market prices change, and the tool approach lets you control exactly what data the LLM receives without exposing your entire database schema
- D. Tools require fewer LLM API calls; and tools are the only way to handle authentication

**5. Which change would most directly fix the LLM's tool selection?**

- A. Add more tools to give the LLM more options
- B. Rewrite the `get_order` description to say 'Use order IDs only, not customer IDs. To find orders for a customer, call lookup_customer first.'
- C. Change the `order_id` parameter type to an integer to prevent the LLM from passing string IDs
- D. Combine both tools into a single `lookup_customer_and_orders(customer_id)` tool

**6. What is the primary cause of the added latency, and what is one architectural approach to reduce it?**

- A. Tool schemas make the prompt longer, increasing LLM processing time; fix by shortening descriptions
- B. Each tool-augmented answer requires two LLM API calls (one to get the tool request, one to get the final answer); reduce latency by caching tool results for frequently asked queries
- C. The dispatch layer adds overhead; fix by calling functions asynchronously
- D. The tool_result JSON adds tokens to the second call; fix by compressing the JSON before sending it

**7. What does this incident reveal about the tradeoff between raw dict schemas and Pydantic-generated schemas?**

- A. Pydantic schemas are less reliable than raw dicts because Pydantic adds extra fields the LLM cannot understand
- B. Pydantic automates schema generation but does not protect against bad naming choices; the field name in the model becomes the parameter name the LLM must use, so naming discipline is still required
- C. Raw dict schemas would have prevented this because they require manual review of every parameter name
- D. The LLM ignores Pydantic-generated schemas because they include JSON Schema keywords like 'title' that confuse it

**8. Which rewrite of the send_email description would most reduce incorrect calls?**

- A. 'Sends an email to a user. Use this tool only when the user explicitly asks to send, draft, or compose an email message.'
- B. 'Sends an email. Do not call this tool unless absolutely necessary.'
- C. 'Communications tool for outbound messages to users.'
- D. 'Use this tool for all user-facing responses that contain important information.'

_Answers are in `checks.json` in the lesson directory._